# TinyStories : la rémanence contre T_rope (XPU / CUDA / CPU)
Amendement 36. Trois cellules : installation, entraînement (≈ 30 min + évaluation et génération), envoi des résultats.
Les deux modèles font **le même nombre de pas** (calculé par une mesure de temps de la rémanence). Si la mémoire de l'appareil est trop petite, le batch est divisé par 2 automatiquement.

In [ ]:
import os, sys, subprocess
if not os.path.exists("src/s3_tasks.py"):
    if not os.path.exists("anendophasia-visual-llm"):
        subprocess.run(["git", "clone", "https://github.com/mathisfrinzi/anendophasia-visual-llm.git"], check=True)
    os.chdir("anendophasia-visual-llm")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "tokenizers"], check=True)
import torch
print("torch", torch.__version__, "| xpu :", hasattr(torch, "xpu") and torch.xpu.is_available(), "| cuda :", torch.cuda.is_available())

In [ ]:
MINUTES = 30      # duree d'entrainement de la remanence (T_rope fait le meme nombre de pas, plus vite)
BATCH   = 256     # divise par 2 automatiquement si la memoire manque
WINDOW  = 64      # longueur des fenetres d'entrainement (l'evaluation va jusqu'a 256)
DATA_MB = 80      # Mo de TinyStories-train utilises (~ 20 M tokens)
cmd = [sys.executable, "-u", "scripts/ts_remanence.py", "--minutes", str(MINUTES), "--batch", str(BATCH), "--window", str(WINDOW), "--data-mb", str(DATA_MB)]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
p.wait()

In [ ]:
# Envoi des resultats (facultatif : necessite d'etre connecte a GitHub) ; sinon, copier les lignes RESULT affichees ci-dessus.
subprocess.run("git add results/ts_*.json results/ts_samples_*.txt && git commit -m 'results/ts (TinyStories, XPU)' && git push", shell=True)